<a href="https://colab.research.google.com/github/mohanad-abdalwahab/-/blob/main/compare_to_final_sheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import re

# =========================================
# 1) الملفات
# =========================================
file1 = "department_employee_cleaned2.xlsx"   # الكشف
file2 = "التسكين الوضع الحالي 2026.xlsx"                  # الهيكل
output_file = "2مسميات_زائدة_في_الكشف.xlsx"

# =========================================
# 2) قراءة البيانات
# =========================================
df1 = pd.read_excel(file1, sheet_name=0)
df2 = pd.read_excel(file2, sheet_name=0)

# =========================================
# 3) تنظيف أسماء الأعمدة
# =========================================
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

# =========================================
# 4) توحيد أسماء الأعمدة
# =========================================
df1 = df1.rename(columns={
    'كود الإدارة': 'كود الإدارة',
    'المسمي الوظيفي': 'المسمى الوظيفي'
})

df2 = df2.rename(columns={
    'رقم  الإدارة': 'كود الإدارة',
    'المسمي الوظيفي': 'المسمى الوظيفي'
})

# =========================================
# 5) اختيار الأعمدة المهمة
# =========================================
df1 = df1[['الإدارة', 'كود الإدارة', 'المسمى الوظيفي']].copy()
df2 = df2[['الإدارة', 'كود الإدارة', 'المسمى الوظيفي']].copy()


# =========================================
# 7) المقارنة (المهم)
# =========================================
# إنشاء مفاتيح مقارنة (كود + مسمى)
keys1 = set(zip(df1['كود الإدارة'], df1['المسمى الوظيفي']))
keys2 = set(zip(df2['كود الإدارة'], df2['المسمى الوظيفي']))

# المسميات الزائدة في الكشف
extra = keys1 - keys2

# =========================================
# 8) تحويل ل DataFrame
# =========================================
result = pd.DataFrame(list(extra), columns=['كود الإدارة', 'المسمى الوظيفي'])

# إضافة اسم الإدارة
dept_names = df1[['كود الإدارة', 'الإدارة']].drop_duplicates()
result = result.merge(dept_names, on='كود الإدارة', how='left')

# ترتيب
result = result[['الإدارة', 'كود الإدارة', 'المسمى الوظيفي']]
result = result.sort_values(by=['الإدارة', 'المسمى الوظيفي'])

# =========================================
# 9) حفظ النتيجة
# =========================================
result.to_excel(output_file, index=False)

print("تم استخراج المسميات الزائدة بنجاح")
print("عدد المسميات:", len(result))

KeyError: "['كود الإدارة'] not in index"

In [2]:
import pandas as pd

# =========================================
# 1) الملفات
# =========================================
file1 = "department_employee_cleaned2.xlsx"   # الكشف
file2 = "هيكل الوظيفي.xlsx"                  # الهيكل
output_file = "21مسميات_زائدة_في_الكشف.xlsx"

# =========================================
# 2) قراءة البيانات
# =========================================
df1 = pd.read_excel(file1, sheet_name=0)
df2 = pd.read_excel(file2, sheet_name=0)

# =========================================
# 3) تنظيف أسماء الأعمدة
# =========================================
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

# =========================================
# 4) توحيد أسماء الأعمدة
# =========================================
df1 = df1.rename(columns={
    'المسمي الوظيفي': 'المسمى الوظيفي'
})

df2 = df2.rename(columns={
    'رقم  الإدارة': 'كود الإدارة',
    'المسمي الوظيفي': 'المسمى الوظيفي'
})

# =========================================
# 5) اختيار الأعمدة المهمة
# =========================================
df1 = df1[['الإدارة', 'كود الإدارة', 'المسمى الوظيفي', 'عدد الموظفين']].copy()
df2 = df2[['الإدارة', 'كود الإدارة', 'المسمى الوظيفي']].copy()

# =========================================
# 6) حذف التكرار من الهيكل
# =========================================
df2 = df2.drop_duplicates(subset=['كود الإدارة', 'المسمى الوظيفي'])

# =========================================
# 7) المقارنة
# =========================================
keys2 = set(zip(df2['كود الإدارة'], df2['المسمى الوظيفي']))

result = df1[
    ~df1[['كود الإدارة', 'المسمى الوظيفي']]
    .apply(tuple, axis=1)
    .isin(keys2)
].copy()

# =========================================
# 8) ترتيب الأعمدة
# =========================================
result = result[
    ['الإدارة', 'كود الإدارة', 'المسمى الوظيفي', 'عدد الموظفين']
]

result = result.sort_values(
    by=['الإدارة', 'المسمى الوظيفي']
).reset_index(drop=True)

# =========================================
# 9) حفظ النتيجة
# =========================================
result.to_excel(output_file, index=False)

print("تم استخراج المسميات الزائدة بنجاح")
print("عدد المسميات:", len(result))
print("إجمالي عدد الموظفين:", result['عدد الموظفين'].sum())

تم استخراج المسميات الزائدة بنجاح
عدد المسميات: 2243
إجمالي عدد الموظفين: 3490


In [11]:
import pandas as pd
import re

mismatch_file = "المسميات الغير متطابقة.xlsx"
employees_file = "التسكين الوضع الحالي 2026.xlsx"
output_file = "كشف_الموظفين_للمسميات_غير_المتطابقة_كل_موظف_صف.xlsx"

mismatch_sheet = 0
employees_sheet = "Data"

def clean_text(value):
    if pd.isna(value):
        return None
    text = str(value).replace("\xa0", " ").replace("_", " ").strip()
    text = re.sub(r"[ـ]+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    if text.lower() in ["", "-", "--", "nan", "none", "null"]:
        return None
    return text

def normalize_arabic(value):
    text = clean_text(value)
    if text is None:
        return None

    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = re.sub(r"[\/\\]+", " ", text)
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_code(value):
    if pd.isna(value):
        return None
    try:
        return str(int(float(value)))
    except Exception:
        return clean_text(value)

def normalize_department(value):
    text = normalize_arabic(value)
    if text is None:
        return None

    fixes = {
        "اداره": "ادارة",
        "شوون": "شؤون",
    }

    text = " ".join(fixes.get(w, w) for w in text.split())
    return re.sub(r"\s+", " ", text).strip()

def normalize_title(value):
    text = normalize_arabic(value)
    if text is None:
        return None

    text = re.sub(r"\bمساعدعامل\b", "مساعد عامل", text)
    text = re.sub(r"\bمساعدمدير\b", "مساعد مدير", text)
    text = re.sub(r"\bخبيرمختبر\b", "خبير مختبر", text)
    text = re.sub(r"\bخبيرشؤون\b", "خبير شؤون", text)
    text = re.sub(r"\bخبيرتفتيش\b", "خبير تفتيش", text)
    text = re.sub(r"\bمفتيش\b", "مفتش", text)
    text = re.sub(r"\bمرافبه\b", "مراقبه", text)
    text = re.sub(r"\bمرا قبه\b", "مراقبه", text)
    text = re.sub(r"\bعامل ملفا ت\b", "عامل ملفات", text)
    text = re.sub(r"\bمدخل بيانات هتقارير\b", "مدخل بيانات وتقارير", text)
    text = re.sub(r"\bمدخل بيانات و تقارير\b", "مدخل بيانات وتقارير", text)

    word_fixes = {
        "رييس": "رئيس",
        "ريس": "رئيس",
        "اسشاري": "استشاري",
        "اخصايي": "اخصائي",
        "اخصائية": "اخصائي",
        "اخصائيه": "اخصائي",
        "ادارية": "اداري",
        "اداريه": "اداري",
        "فنية": "فني",
        "فنيه": "فني",
        "مهندسه": "مهندس",
        "ميكانيكي": "ميكانيكا",
        "ميكانيكيه": "ميكانيكا",
        "ميكانيك": "ميكانيكا",
        "كهربائي": "كهرباء",
        "كهربائيه": "كهرباء",
        "نطافه": "نظافه",
        "ملاجظ": "ملاحظ",
        "متايعه": "متابعه",
        "تفيش": "تفتيش",
        "بييي": "بيئه",
        "بيييه": "بيئه",
        "مختبيريه": "مختبريه",
        "محتبريه": "مختبريه",
        "تطبقات": "تطبيقات",
        "وتطبقات": "وتطبيقات",
        "تربييه": "تربيه",
        "وقواراض": "وقوارض",
        "وقواض": "وقوارض",
        "شوون": "شؤون",
        "سفيه": "سفينه",
        "فزيايي": "فيزيائي",
        "فيزيايي": "فيزيائي",
        "نضام": "نظام",
    }

    text = " ".join(word_fixes.get(w, w) for w in text.split())

    phrase_fixes = {
        "مدخل بيانات تقارير": "مدخل بيانات وتقارير",
        "عامل تشحيم معدات ساس": "عامل تشحيم معدات سادس",
        "عامل زراعي ساس": "عامل زراعي سادس",
        "اخصائي مهندس ميكانيكا اول": "مهندس ميكانيكا اول",
        "اخصائي مهندس جوده مشاريع اول": "اخصائي هندسي جوده مشاريع اول",
        "كبير مهندسين ميكانيكي": "كبير مهندسين ميكانيكا",
    }

    for old, new in phrase_fixes.items():
        text = text.replace(old, new)

    text = re.sub(r"بالتكل$", "بالتكليف", text)
    text = re.sub(r"بالتك$", "بالتكليف", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def find_column(columns, candidates):
    normalized_columns = {col: normalize_arabic(col) for col in columns}
    normalized_candidates = [normalize_arabic(c) for c in candidates]

    for original, normalized in normalized_columns.items():
        if normalized in normalized_candidates:
            return original

    for original, normalized in normalized_columns.items():
        for candidate in normalized_candidates:
            if candidate and normalized and (candidate in normalized or normalized in candidate):
                return original

    return None

def main():
    df_mismatch = pd.read_excel(mismatch_file, sheet_name=mismatch_sheet)
    df_emp = pd.read_excel(employees_file, sheet_name=employees_sheet)

    df_mismatch.columns = df_mismatch.columns.astype(str).str.strip()
    df_emp.columns = df_emp.columns.astype(str).str.strip()

    mismatch_dept_col = find_column(df_mismatch.columns, ["الإدارة", "الادارة", "اسم الإدارة", "اسم الادارة"])
    mismatch_code_col = find_column(df_mismatch.columns, ["كود الإدارة", "كود الادارة", "رقم الإدارة", "رقم الادارة"])
    mismatch_title_col = find_column(df_mismatch.columns, ["المسمى الوظيفي", "المسمي الوظيفي", "المسمى", "المسمي"])

    emp_name_col = find_column(df_emp.columns, ["الاسم", "اسم الموظف", "الموظف"])
    emp_no_col = find_column(df_emp.columns, ["رقم الموظف", "رقم موظف"])
    emp_code_col = find_column(df_emp.columns, ["الإدارة- code", "كود الإدارة", "كود الادارة", "رقم الإدارة", "رقم الادارة"])
    emp_title_col = find_column(df_emp.columns, ["المسمى الوظيفي", "المسمي الوظيفي", "المسمى", "المسمي"])

    missing = []
    checks = {
        "الإدارة في ملف المسميات الغير متطابقة": mismatch_dept_col,
        "كود الإدارة في ملف المسميات الغير متطابقة": mismatch_code_col,
        "المسمى الوظيفي في ملف المسميات الغير متطابقة": mismatch_title_col,
        "اسم الموظف في ملف التسكين": emp_name_col,
        "رقم الموظف في ملف التسكين": emp_no_col,
        "كود الإدارة في ملف التسكين": emp_code_col,
        "المسمى الوظيفي في ملف التسكين": emp_title_col,
    }

    for label, col in checks.items():
        if col is None:
            missing.append(label)

    if missing:
        raise ValueError("الأعمدة التالية غير موجودة:\n- " + "\n- ".join(missing))

    mismatch = pd.DataFrame({
        "الإدارة": df_mismatch[mismatch_dept_col].apply(normalize_department),
        "كود الإدارة": df_mismatch[mismatch_code_col].apply(normalize_code),
        "المسمى الوظيفي": df_mismatch[mismatch_title_col].apply(clean_text),
    })

    mismatch["مفتاح المسمى"] = mismatch["المسمى الوظيفي"].apply(normalize_title)

    mismatch = mismatch.dropna(subset=["كود الإدارة", "مفتاح المسمى"])

    mismatch = (
        mismatch
        .groupby(["كود الإدارة", "مفتاح المسمى"], as_index=False)
        .agg(
            الإدارة=("الإدارة", "first"),
            المسمى_الوظيفي=("المسمى الوظيفي", "first"),
            صيغ_المسمى_في_ملف_المسميات=("المسمى الوظيفي", lambda x: "، ".join(sorted(set(map(str, x)))))
        )
    )

    employees = pd.DataFrame({
        "رقم الموظف": df_emp[emp_no_col].apply(normalize_code),
        "اسم الموظف": df_emp[emp_name_col].apply(clean_text),
        "كود الإدارة": df_emp[emp_code_col].apply(normalize_code),
        "المسمى في التسكين": df_emp[emp_title_col].apply(clean_text),
    })

    employees["مفتاح المسمى"] = employees["المسمى في التسكين"].apply(normalize_title)

    employees = employees.dropna(
        subset=["رقم الموظف", "اسم الموظف", "كود الإدارة", "مفتاح المسمى"]
    )

    employees = employees.drop_duplicates(
        subset=["رقم الموظف", "كود الإدارة", "مفتاح المسمى"]
    )

    result = mismatch.merge(
        employees,
        on=["كود الإدارة", "مفتاح المسمى"],
        how="left"
    )

    result = result.rename(columns={
        "المسمى_الوظيفي": "المسمى الوظيفي"
    })

    result = result[
        [
            "الإدارة",
            "كود الإدارة",
            "المسمى الوظيفي",
            "رقم الموظف",
            "اسم الموظف",
            "المسمى في التسكين",
            "صيغ_المسمى_في_ملف_المسميات",
            "مفتاح المسمى",
        ]
    ]

    result = result.sort_values(
        ["الإدارة", "المسمى الوظيفي", "اسم الموظف"],
        na_position="last"
    ).reset_index(drop=True)

    no_match = result[result["اسم الموظف"].isna()].copy()

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        result.to_excel(writer, sheet_name="الكشف_النهائي", index=False)
        no_match.to_excel(writer, sheet_name="مسميات_بدون_موظفين", index=False)
        mismatch.to_excel(writer, sheet_name="المسميات_بعد_التوحيد", index=False)
        employees.to_excel(writer, sheet_name="التسكين_بعد_التوحيد", index=False)

    print("تم إنشاء الملف بنجاح:", output_file)
    print("عدد الصفوف في الكشف:", len(result))
    print("عدد الموظفين:", result["رقم الموظف"].nunique())
    print("عدد المسميات بدون موظفين:", len(no_match))

if __name__ == "__main__":
    main()

تم إنشاء الملف بنجاح: كشف_الموظفين_للمسميات_غير_المتطابقة_كل_موظف_صف.xlsx
عدد الصفوف في الكشف: 3249
عدد الموظفين: 3247
عدد المسميات بدون موظفين: 2
